# Qwen LoRA Fine-Tuning in Google Colab

Upload the project ZIP, unpack it, install the training dependencies, and fine-tune `Qwen/Qwen2.5-0.5B-Instruct` on the included email security dataset.

Use a Colab GPU runtime (`Runtime > Change runtime type > T4 GPU` or better) before running the training cell. The default configuration uses the complete dataset; set `LIMIT` to a small number for a smoke test.

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_names:
    raise FileNotFoundError('Upload the AI_Gaurdrails_colab.zip file.')
zip_name = zip_names[0]
print(f'Uploaded: {zip_name}')

In [ ]:
from pathlib import Path
import shutil
import zipfile

project_dir = Path('/content/AI_Gaurdrails')
if project_dir.exists():
    shutil.rmtree(project_dir)
project_dir.mkdir(parents=True)

with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(project_dir)

# Support archives containing either project files or one top-level folder.
dataset_candidates = list(project_dir.rglob('data/email_agent_security_dataset.xlsx'))
if not dataset_candidates:
    raise FileNotFoundError('The uploaded ZIP does not contain data/email_agent_security_dataset.xlsx.')
repo_root = dataset_candidates[0].parents[1]
print(f'Project root: {repo_root}')
print(f'Dataset: {dataset_candidates[0]}')

In [ ]:
# This is text-only training; torchvision is not needed and can conflict with Colab's torch build.
%pip -q uninstall -y torchvision
%pip -q install --upgrade \
    'transformers==4.46.3' \
    'peft==0.13.2' \
    'datasets==2.21.0' \
    'accelerate==0.34.2' \
    sentencepiece

import sys
sys.path.insert(0, str(repo_root))

from common.dataset import DATA_SHEETS, load_actions
import torch
import transformers
import peft

actions = load_actions(str(dataset_candidates[0]))
print(f'Loaded {len(actions):,} training records from {len(DATA_SHEETS)} dataset sheets.')
print(f'GPU available: {torch.cuda.is_available()}')
print(f'transformers={transformers.__version__}, peft={peft.__version__}')

In [ ]:
# Training configuration. Set LIMIT to an integer for a smoke test, or None for all records.
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
EPOCHS = 1
BATCH_SIZE = 2
LIMIT = None
MAX_LENGTH = 512
LEARNING_RATE = 2e-4
SEED = 0
OUTPUT_DIR = str(repo_root / 'classifiers' / 'results' / 'qwen_lora_colab')

print({
    'model': MODEL_ID,
    'examples': len(actions) if LIMIT is None else min(LIMIT, len(actions)),
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'output_dir': OUTPUT_DIR,
})

In [ ]:
from classifiers.qwen.qwen_finetune import train_and_evaluate

summary = train_and_evaluate(
    model_id=MODEL_ID,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    limit=LIMIT,
    output_dir=OUTPUT_DIR,
    max_length=MAX_LENGTH,
    learning_rate=LEARNING_RATE,
    seed=SEED,
)
summary

In [ ]:
from google.colab import files
from pathlib import Path
import json
import shutil
import zipfile
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Use the adapter produced by training, or upload a previously downloaded adapter ZIP.
training_summary = globals().get('summary')
adapter_dir = Path(training_summary.get('adapter_dir', OUTPUT_DIR)) if isinstance(training_summary, dict) else Path(OUTPUT_DIR)
if not adapter_dir.is_dir():
    print('Training output not found. Upload qwen_lora_colab_output.zip.')
    uploaded_adapter = files.upload()
    adapter_zip = next(
        name for name in uploaded_adapter
        if name.lower().endswith('.zip')
    )
    adapter_root = Path('/content/qwen_lora_imported')
    if adapter_root.exists():
        shutil.rmtree(adapter_root)
    adapter_root.mkdir(parents=True)
    with zipfile.ZipFile(adapter_zip) as archive:
        archive.extractall(adapter_root)
    adapter_configs = list(adapter_root.rglob('adapter_config.json'))
    if len(adapter_configs) != 1:
        raise FileNotFoundError('Expected exactly one adapter_config.json in the uploaded ZIP.')
    adapter_dir = adapter_configs[0].parent

print(f'Using adapter: {adapter_dir}')

inference_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
model = PeftModel.from_pretrained(base_model, adapter_dir)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

from classifiers.qwen.qwen_finetune import prepare_dataset
examples = prepare_dataset(actions, LIMIT, SEED)
split_at = max(1, int(len(examples) * 0.9))
validation_examples = examples[split_at:] or examples[:1]
predictions = []

for item in validation_examples:
    prompt = item['text'] + '\nAnswer:'
    inputs = inference_tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=4,
            do_sample=False,
            pad_token_id=inference_tokenizer.pad_token_id,
        )
    generated = inference_tokenizer.decode(
        output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    ).strip().upper()
    prediction = 'ATTACK' if 'ATTACK' in generated else 'BENIGN'
    predictions.append({
        'expected': item['label'],
        'predicted': prediction,
        'raw_output': generated,
    })

accuracy = sum(row['expected'] == row['predicted'] for row in predictions) / len(predictions)
evaluation = {
    'model': MODEL_ID,
    'adapter': str(adapter_dir),
    'device': device,
    'validation_examples': len(predictions),
    'accuracy': accuracy,
    'predictions': predictions,
}
evaluation_path = Path('/content/qwen_validation_results.json')
evaluation_path.write_text(json.dumps(evaluation, indent=2))

print(f'Validation examples: {len(predictions)}')
print(f'Validation accuracy: {accuracy:.3f}')
print(f'Saved results: {evaluation_path}')

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

if not Path('/content/qwen_validation_results.json').is_file():
    raise RuntimeError('Run the training/evaluation cell before downloading results.')

results_path = Path('/content/qwen_validation_results.json')
files.download(str(results_path))
print(f'Downloaded evaluation results: {results_path}')

adapter_dir = Path(summary.get('adapter_dir', OUTPUT_DIR)) if isinstance(summary, dict) else Path('/content/qwen_lora_imported')
if not adapter_dir.is_dir():
    raise FileNotFoundError(f'Adapter directory not found: {adapter_dir}')

download_zip = Path('/content/qwen_lora_colab_output.zip')
if download_zip.exists():
    download_zip.unlink()
shutil.make_archive(str(download_zip.with_suffix('')), 'zip', adapter_dir)
files.download(str(download_zip))
print(f'Downloaded adapter archive: {download_zip}')